In [1]:
import random
from collections import deque

In [2]:
RANKS = "23456789TJQKA"
SUITS = "♠♥♣♦"
SMALL_BLIND = 1
BIG_BLIND = 2


def newDeck():
    deck = deque(f"{rank}{suit}" for rank in RANKS for suit in SUITS)
    random.shuffle(deck)
    return deck

In [3]:
class Player:
    def __init__(self, name, chips):
        self.name = name
        self.chips = chips
        self.hand = []
        self.hasActedThisRound = False
        self.currentBet = 0
        self.isFolded = False
        self.isAllIn = False

    def resetHand(self):
        self.hand = []
        self.hasActedThisRound = False
        self.currentBet = 0
        self.isFolded = False
        self.isAllIn = False

In [4]:
class PokerGame:
    def __init__(self, players):
        self.players = [Player(name, chips) for name, chips in players]

        self.smallBlind = SMALL_BLIND
        self.bigBlind = BIG_BLIND

        self.deck = []
        self.communityCards = []

        self.pot = 0
        self.currentPlayerPosition = 0

        self.gameState = "null"
        self.roundMaxBet = 0
        self.minimumRaiseSize = self.bigBlind

################################################################################
### Formatting
################################################################################
    def printPlayerStatus(self):
        print("PLAYER STATUS")
        print("Name                 | Chips    | Bet      | Hand")
        print("---------------------+----------+----------+----------------")
        for player in self.players:
            print(f"{player.name:<20} | ${player.chips:<7} | ${player.currentBet:<7} | {player.hand}")
        print("---------------------+----------+----------+----------------")

################################################################################
### Convenience Functions/Properties
################################################################################
    @property
    def currentPlayer(self):
        return self.players[self.currentPlayerPosition]

    def advancePlayer(self):
        self.currentPlayerPosition = (self.currentPlayerPosition + 1) % len(self.players)
    @property
    def activePlayers(self):
        return [player for player in self.players if not player.isFolded]

################################################################################
### Turns and Turn Validation
################################################################################
    def checkValidBet(self, player, bet):
        if bet == -1:
            player.isFolded = True
            return True
        
        if bet < 0:
            print(f"Invalid bet, please enter a positive number.")
            return False

        if bet == player.chips and bet  > 0:
            player.isAllIn = True
            return True
        
        if bet > player.chips:
            print(f"Invalid bet, please put exact chip value for all-in.")
            return False
        
        if player.currentBet + bet == self.roundMaxBet:
            print(f"{player} calls.")
            return True
        
        if player.currentBet + bet < self.roundMaxBet:
            print(f"Invalid bet.")
            return False
        
        if player.currentBet + bet < self.roundMaxBet + self.minimumRaiseSize:
            print(f"Invalid raise, minimum raise is ${self.minimumRaiseSize}, you to enter at least ${self.roundMaxBet + self.minimumRaiseSize - player.currentBet} to raise.")
            return False
        
        return True

    def getPlayerAction(self, player):
        print(f"Current bet to call is ${self.roundMaxBet - player.currentBet}")
        while True:
            try:
                bet = int(input(f"{player.name}, enter your bet, -1 to fold: "))
            except ValueError:
                continue

            if self.checkValidBet(player, bet):
                return bet

    def playerTurn(self):
        player = self.currentPlayer

        if (player.isFolded or player.isAllIn):
            self.advancePlayer()
            return

        print(f"\n{player.name}'s turn")
        self.printPlayerStatus()
        print(f"Player hand: {player.hand} | Community cards: {self.communityCards}")

        bet = self.getPlayerAction(player)

        # If folded this turn
        if bet == -1:
            player.hasActedThisRound = True
            self.advancePlayer()
            return

        # Apply their bet
        player.chips -= bet
        player.currentBet += bet
        self.pot += bet
        player.hasActedThisRound = True

        # If they increased the maximum bet
        if player.currentBet > self.roundMaxBet:
            raiseAmount = player.currentBet - self.roundMaxBet

            # Full raise - reopen betting and update minimum raise size
            if raiseAmount >= self.minimumRaiseSize:
                self.minimumRaiseSize = raiseAmount

                for p in self.players:
                    if not p.isFolded and not p.isAllIn:
                        p.hasActedThisRound = False

                player.hasActedThisRound = True

            # Any raise updates the amount to call
            self.roundMaxBet = player.currentBet

        self.advancePlayer()
################################################################################
### Preparing and Ending a Round
################################################################################
    def resetBettingRound(self):
        self.roundMaxBet = 0
        self.minimumRaiseSize = self.bigBlind
        self.currentPlayerPosition = 0
        for player in self.players:
            player.currentBet = 0
            if ((not player.isAllIn) and (not player.isFolded)):
                player.hasActedThisRound = False

    def isBettingRoundComplete(self):
        if len(self.activePlayers) <= 1:
            return True
        
        if not all(player.hasActedThisRound for player in self.activePlayers):
            return False
        
        return all(((player.currentBet == self.roundMaxBet) or player.isAllIn) for player in self.activePlayers)

    def bettingRound(self):
        while not self.isBettingRoundComplete():
            self.playerTurn()

        print("Betting round complete.")

################################################################################
### Dealing and Blinds (Forced Actions)
################################################################################
    def burnCard(self):
        self.deck.popleft()

    def dealHoleCards(self):
        self.resetBettingRound()
        for _ in range(2):
            for player in self.players:
                player.hand.append(self.deck.popleft())

    def postBlinds(self):
        sb = self.players[0]
        bb = self.players[1]

        if sb.chips >= self.smallBlind:
            sb.chips -= self.smallBlind
            sb.currentBet = self.smallBlind
            print(f"Player {sb.name} posted small blind of {self.smallBlind}.")
            self.advancePlayer()
        else:
            sb.currentBet = sb.chips
            print(f"Player {sb.name} posted an all-in small blind of {sb.chips}.")
            sb.chips = 0
            sb.isAllIn = True
            self.advancePlayer()

        if bb.chips >= self.bigBlind:
            bb.chips -= self.bigBlind
            bb.currentBet = self.bigBlind
            print(f"Player {bb.name} posted big blind of {self.bigBlind}.")
            self.minimumRaiseSize = self.bigBlind
            self.advancePlayer()
        else:
            bb.currentBet = bb.chips
            print(f"Player {bb.name} posted an all-in big blind of {bb.chips}.")
            bb.chips = 0
            bb.isAllIn = True
            self.advancePlayer()

        self.pot += sb.currentBet + bb.currentBet
        self.roundMaxBet = max(sb.currentBet, bb.currentBet)

    def dealFlop(self):
        self.resetBettingRound()
        self.burnCard()
        self.communityCards.extend(self.deck.popleft() for _ in range(3))     

    def dealTurn(self):
        self.resetBettingRound()
        self.burnCard()
        self.communityCards.append(self.deck.popleft())

    def dealRiver(self):
        self.resetBettingRound()
        self.burnCard()
        self.communityCards.append(self.deck.popleft())    

################################################################################
### Container Function for Each Round
################################################################################
    def startHand(self):
        self.deck = newDeck()
        self.communityCards = []
        self.pot = 0
        self.minimumRaiseSize = self.bigBlind

        for player in self.players:
            player.resetHand()

        self.dealHoleCards()
        self.postBlinds()
        self.gameState = "Pre-flop"
        self.bettingRound()

        self.dealFlop()
        self.gameState = "Flop"
        self.bettingRound()

        self.dealTurn()
        self.gameState = "Turn"
        self.bettingRound()

        self.dealRiver()
        self.gameState = "River"
        self.bettingRound()

        print(f"Pot size: {self.pot}")

        # outputs the hands in the format for handEvaluator
        holeCards = []
        for player in self.players:
            holeCards.append(player.hand)

        print(holeCards)

        print(self.communityCards)

In [5]:
game = PokerGame([("Player1", 1000), ("Player2", 1000), ("Player3", 1000)])
game.startHand()



Player Player1 posted small blind of 1.
Player Player2 posted big blind of 2.

Player3's turn
PLAYER STATUS
Name                 | Chips    | Bet      | Hand
---------------------+----------+----------+----------------
Player1              | $999     | $1       | ['K♠', '3♣']
Player2              | $998     | $2       | ['9♠', '9♥']
Player3              | $1000    | $0       | ['9♣', 'J♣']
---------------------+----------+----------+----------------
Player hand: ['9♣', 'J♣'] | Community cards: []
Current bet to call is $2
<__main__.Player object at 0x7d82b36d1e80> calls.

Player1's turn
PLAYER STATUS
Name                 | Chips    | Bet      | Hand
---------------------+----------+----------+----------------
Player1              | $999     | $1       | ['K♠', '3♣']
Player2              | $998     | $2       | ['9♠', '9♥']
Player3              | $998     | $2       | ['9♣', 'J♣']
---------------------+----------+----------+----------------
Player hand: ['K♠', '3♣'] | Community cards: [